# 02 — Threshold Sweep

Run `stabstream-threshold` across a range of physical error rates and distances,
then visualize the p_L vs p crossing to estimate the code threshold.

**Requirements:** `pip install stabstream numpy matplotlib`  
**Optional:** `stabstream-threshold` binary in PATH (cargo install stabstream-threshold),
and `.dem` files for each distance. Synthetic data is used automatically if unavailable.

In [ ]:
import json
import subprocess
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Configuration ---
DEM_FILES = {
    3: None,  # e.g. "../models/surface_d3.dem"
    5: None,  # e.g. "../models/surface_d5.dem"
    7: None,  # e.g. "../models/surface_d7.dem"
}
P_PHYSICAL_VALUES = [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.013, 0.016, 0.020]
SHOTS = 50_000
DECODER = "union-find"  # or "null"
OUTPUT_JSON = Path("threshold_results.json")

## 1. Run threshold sweep (or load cached results)

In [ ]:
def run_threshold_sweep(
    dem_files: dict[int, str],
    p_values: list[float],
    shots: int,
    decoder: str,
    output: Path,
) -> list[dict]:
    """Run stabstream-threshold and return parsed results."""
    if not shutil.which("stabstream-threshold"):
        raise FileNotFoundError(
            "stabstream-threshold not found. "
            "Build with: cargo install --path crates/stabstream-threshold"
        )
    cmd = ["stabstream-threshold", "run"]
    for p in p_values:
        cmd += ["--p-physical", str(p)]
    for dem in dem_files.values():
        if dem:
            cmd += ["--dem", dem]
    cmd += ["--shots", str(shots), "--decoder", decoder, "--out", str(output)]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    print(result.stdout)
    return json.loads(output.read_text())


def synthetic_threshold_data(
    distances: list[int],
    p_values: list[float],
    threshold: float = 0.009,
    seed: int = 42,
) -> list[dict]:
    """Generate plausible synthetic threshold data for demonstration."""
    rng = np.random.default_rng(seed)
    rows = []
    for d in distances:
        for p in p_values:
            # Below threshold: p_L decreases with d. Above: increases.
            if p < threshold:
                p_l = 0.5 * (p / threshold) ** ((d + 1) / 2)
            else:
                p_l = 0.5 * (1 - (1 - p / threshold) ** ((d + 1) / 2))
            p_l = float(np.clip(p_l + rng.normal(0, 0.003), 0, 0.5))
            rows.append({
                "distance": d,
                "p_physical": p,
                "p_l": p_l,
                "p_l_err": float(np.sqrt(p_l * (1 - p_l) / 50_000)),
                "shots": 50_000,
                "logical_errors": int(p_l * 50_000),
            })
    return rows


dem_available = any(
    path and Path(path).exists() for path in DEM_FILES.values()
)

if OUTPUT_JSON.exists():
    print(f"Loading cached results from {OUTPUT_JSON}")
    data = json.loads(OUTPUT_JSON.read_text())
elif dem_available and shutil.which("stabstream-threshold"):
    dem_paths = {d: p for d, p in DEM_FILES.items() if p and Path(p).exists()}
    data = run_threshold_sweep(dem_paths, P_PHYSICAL_VALUES, SHOTS, DECODER, OUTPUT_JSON)
else:
    print("stabstream-threshold or DEM files not found — using synthetic data.")
    data = synthetic_threshold_data(
        distances=list(DEM_FILES.keys()),
        p_values=P_PHYSICAL_VALUES,
    )

print(f"Loaded {len(data)} data points")

## 2. Plot threshold curves

In [ ]:
from stabstream.plot import plot_threshold_curves

fig, ax = plt.subplots(figsize=(8, 5))
plot_threshold_curves(
    data,
    ax=ax,
    title=f"QEC Threshold ({DECODER} decoder)",
    show_diagonal=True,
)
plt.tight_layout()
plt.savefig("threshold_curves.png", dpi=150)
plt.show()

## 3. Estimate threshold by curve crossing

In [ ]:
from collections import defaultdict


def estimate_threshold(data: list[dict]) -> float | None:
    """Find the crossing point between adjacent-distance curves (linear interpolation)."""
    by_dist: dict[int, list] = defaultdict(list)
    for pt in data:
        by_dist[pt["distance"]].append(pt)
    for dist in by_dist:
        by_dist[dist].sort(key=lambda p: p["p_physical"])

    dists = sorted(by_dist.keys())
    if len(dists) < 2:
        return None

    crossings = []
    for d_lo, d_hi in zip(dists, dists[1:]):
        pts_lo = by_dist[d_lo]
        pts_hi = by_dist[d_hi]
        # Interpolate at the same p_physical values
        ps = sorted(set(p["p_physical"] for p in pts_lo) &
                    set(p["p_physical"] for p in pts_hi))
        lo_map = {p["p_physical"]: p["p_l"] for p in pts_lo}
        hi_map = {p["p_physical"]: p["p_l"] for p in pts_hi}
        # Find sign change in (p_L_hi - p_L_lo)
        diffs = [(p, hi_map[p] - lo_map[p]) for p in ps]
        for (p0, d0), (p1, d1) in zip(diffs, diffs[1:]):
            if d0 * d1 < 0:  # sign change
                # linear interpolation
                p_cross = p0 - d0 * (p1 - p0) / (d1 - d0)
                crossings.append(p_cross)

    return float(np.mean(crossings)) if crossings else None


p_threshold = estimate_threshold(data)

if p_threshold is not None:
    print(f"Estimated threshold: p_th ≈ {p_threshold:.4f} ({p_threshold*100:.2f}%)")
    # Re-plot with threshold line
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_threshold_curves(data, ax=ax,
                          title=f"QEC Threshold  (p_th ≈ {p_threshold:.4f})")
    ax.axvline(p_threshold, color="black", linestyle=":", linewidth=2, alpha=0.7,
               label=f"Threshold ≈ {p_threshold:.4f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig("threshold_curves_annotated.png", dpi=150)
    plt.show()
else:
    print("Could not find a crossing — try more distances or extend p range.")

## 4. Code distance scaling below threshold

Below threshold, p_L should scale as `(p/p_th)^((d+1)/2)`. Verify this on a log-log plot.

In [ ]:
from collections import defaultdict

by_dist: dict[int, list] = defaultdict(list)
for pt in data:
    by_dist[pt["distance"]].append(pt)

# Use the p value closest to half the estimated threshold
p_ref = (p_threshold / 2) if p_threshold else P_PHYSICAL_VALUES[2]
p_ref_key = min(P_PHYSICAL_VALUES, key=lambda p: abs(p - p_ref))

distances, p_ls, p_l_errs = [], [], []
for d in sorted(by_dist.keys()):
    pts = [p for p in by_dist[d] if p["p_physical"] == p_ref_key]
    if pts:
        distances.append(d)
        p_ls.append(pts[0]["p_l"])
        p_l_errs.append(pts[0].get("p_l_err", 0))

if len(distances) >= 2:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(distances, p_ls, yerr=p_l_errs, fmt="-o", color="#4363D8",
                capsize=4, label=f"p = {p_ref_key}")
    ax.set_yscale("log")
    ax.set_xlabel("Code distance  d")
    ax.set_ylabel("Logical error rate  p_L")
    ax.set_title(f"p_L vs distance at p = {p_ref_key}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("distance_scaling.png", dpi=150)
    plt.show()

    # Fit suppression factor λ where p_L ∝ λ^d
    if all(x > 0 for x in p_ls):
        log_pls = np.log(p_ls)
        coeff = np.polyfit(distances, log_pls, 1)
        lam = np.exp(coeff[0])
        print(f"Suppression factor λ ≈ {lam:.3f}  (p_L ∝ λ^d, λ < 1 means suppressed)")
else:
    print("Need ≥ 2 distances for scaling plot.")